# Resume-JD Deep Neural Network — Siamese BiLSTM Match Scorer

This Colab notebook trains a **Deep Neural Network** that reads a **resume**
and a **job description** and predicts whether the resume is a **Weak (0)**,
**Medium (1)**, or **Strong (2)** match for the role.

It implements the architecture described in
`deep_neural_network_resume_jd_in_depth.docx`:

* Text cleaning + tokenization + padding
* A shared **Embedding → BiLSTM → GlobalMaxPooling** encoder tower
  (**Siamese network** — same weights used for both resume and JD)
* Comparison features: `[u, v, |u - v|, u * v]`
* `Dense(128) → Dropout(0.3) → Dense(64) → Dense(3, softmax)` classifier
* `sparse_categorical_crossentropy` loss, `Adam` optimizer
* Evaluation with accuracy / precision / recall / F1 / confusion matrix
* A prediction + explainability demo at the end

**Run in Google Colab:** `Runtime → Run all`. GPU is optional (the model is
small enough to train on CPU in a few minutes).


## 1. Setup

In [ ]:
# If running in Google Colab, uncomment the line below to mount Drive
# and adjust PROJECT_DIR to where you've uploaded this project folder.
#
# from google.colab import drive
# drive.mount('/content/drive')

import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
print("Running in Colab:", IN_COLAB)

# Path to the project root. When you upload/clone the whole
# `resume_jd_dnn_project` folder into Colab (e.g. via the Files pane or
# `git clone`), just set this to that folder.
PROJECT_DIR = "."  # change to e.g. "/content/resume_jd_dnn_project" in Colab
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Install dependencies (Colab already has TensorFlow, pandas, sklearn, matplotlib
# preinstalled — this is here for completeness / non-Colab environments).
!pip install -q -r requirements.txt 2>/dev/null || pip install -q tensorflow-cpu pandas scikit-learn matplotlib


In [ ]:
import sys
sys.path.insert(0, "src")

import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

from preprocessing import build_dataset, clean_text
from model import ModelConfig, build_siamese_model
import train as train_module
import evaluate as evaluate_module
import predict as predict_module
import explain as explain_module

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)


## 2. Build the labeled dataset from the raw Resume/JD JSONL

The raw dataset (`data/raw/train_sample.jsonl`) contains, per record, a job
description, a resume written to **strongly match** it (`Resume-matched`),
a resume written to **weakly match** it (`Resume-unmatched`), and a small
set of `Filtered-information` edit instructions. We use those instructions
to synthesize a **Medium**-match resume (a slightly degraded version of the
strong-match resume), giving us a naturally balanced 3-class dataset.
See `src/preprocessing.py` for the full logic.


In [ ]:
RAW_PATH = Path("data/raw/train_sample.jsonl")
PROCESSED_PATH = Path("data/processed/resume_jd_dataset.csv")

if PROCESSED_PATH.exists():
    df = pd.read_csv(PROCESSED_PATH)
    print(f"Loaded cached processed dataset: {len(df):,} rows")
else:
    df = build_dataset(RAW_PATH, min_words=15)
    PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED_PATH, index=False)
    print(f"Built and cached processed dataset: {len(df):,} rows")

df.head()


In [ ]:
label_names = {0: "Weak", 1: "Medium", 2: "Strong"}
counts = df["match_label"].map(label_names).value_counts()
print(counts)

counts.plot(kind="bar", color=["#e07a5f", "#f2cc8f", "#81b29a"])
plt.title("Class balance: Weak / Medium / Strong")
plt.ylabel("count")
plt.tight_layout()
plt.show()


## 3. Train / validation / test split + tokenization

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["match_label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["match_label"], random_state=42
)
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

test_df.to_csv("data/processed/test_split.csv", index=False)


In [ ]:
VOCAB_SIZE = 15000
MAX_LEN = 120

tokenizer = train_module.fit_tokenizer(
    pd.concat([train_df["resume_text"], train_df["job_description"]]), VOCAB_SIZE
)

def encode_split(d):
    return (
        train_module.encode(tokenizer, d["resume_text"], MAX_LEN),
        train_module.encode(tokenizer, d["job_description"], MAX_LEN),
        d["match_label"].to_numpy(),
    )

X_res_train, X_jd_train, y_train = encode_split(train_df)
X_res_val, X_jd_val, y_val = encode_split(val_df)
X_res_test, X_jd_test, y_test = encode_split(test_df)

print("Vocabulary size used:", min(VOCAB_SIZE, len(tokenizer.word_index) + 1))
print("Resume token shape:", X_res_train.shape)


## 4. Build the Siamese BiLSTM model

Two towers (resume + JD) **share the same encoder weights** — this is what
makes it a *Siamese* network. The comparison layer combines both vectors
with concatenation, absolute difference, and element-wise product before
the final softmax classifier.


In [ ]:
config = ModelConfig(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    embed_dim=96,
    lstm_units=48,
    dense1_units=128,
    dense2_units=64,
    dropout=0.3,
    num_classes=3,
    learning_rate=1e-3,
)

model = build_siamese_model(config)
model.summary()


In [ ]:
try:
    tf.keras.utils.plot_model(
        model, to_file="outputs/model_architecture.png",
        show_shapes=True, show_layer_names=True, rankdir="TB",
    )
    from IPython.display import Image, display
    display(Image("outputs/model_architecture.png"))
except ImportError:
    # pydot/graphviz not installed in this environment - skip the diagram,
    # model.summary() above already shows the full architecture.
    print("(Skipping architecture diagram - install pydot + graphviz to enable it.)")


## 5. Train the model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    {"resume_tokens": X_res_train, "jd_tokens": X_jd_train},
    y_train,
    validation_data=({"resume_tokens": X_res_val, "jd_tokens": X_jd_val}, y_val),
    epochs=15,
    batch_size=64,
    callbacks=callbacks,
    verbose=2,
)


In [ ]:
Path("models").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

model.save("models/resume_jd_match_model.keras")

with open("models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("models/model_config.json", "w") as f:
    json.dump(config.__dict__, f, indent=2)

print("Saved model, tokenizer, and config to models/")


## 6. Training curves

In [ ]:
train_module.plot_history(history, Path("outputs/training_history.png"))

from IPython.display import Image, display
display(Image("outputs/training_history.png"))


## 7. Evaluate on the held-out test set

In [ ]:
probs = model.predict({"resume_tokens": X_res_test, "jd_tokens": X_jd_test}, verbose=0)
y_pred = np.argmax(probs, axis=1)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1 (macro): {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Weak", "Medium", "Strong"], zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
evaluate_module.plot_confusion_matrix(cm, Path("outputs/confusion_matrix.png"))

metrics = {
    "accuracy": acc, "precision_macro": precision, "recall_macro": recall,
    "f1_macro": f1, "confusion_matrix": cm.tolist(),
    "n_test_examples": int(len(test_df)),
}
with open("outputs/evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

from IPython.display import Image, display
display(Image("outputs/confusion_matrix.png"))


## 8. Try a prediction

This mirrors `src/predict.py`: given a resume and a JD, the model returns a
predicted class + confidence, plus a simple common/missing-skills
diagnostic.


In [ ]:
sample_resume = (
    "Python machine learning engineer with 3 years experience building "
    "FastAPI microservices, Docker containers, and deploying models on AWS. "
    "Built a chatbot using embeddings and vector search."
)
sample_jd = (
    "Looking for an AI Engineer with Python, FastAPI, Docker experience "
    "who can build RAG pipelines using a vector database and Kubernetes "
    "for deployment."
)

result = predict_module.predict_match(model, tokenizer, MAX_LEN, sample_resume, sample_jd)
report = predict_module.format_report(result)
print(report)

Path("outputs").mkdir(exist_ok=True)
with open("outputs/prediction_result.md", "w") as f:
    f.write("# Resume-JD Match Prediction\n\n```\n" + report + "\n```\n")


## 9. Explainability: which words drove the prediction?

We use a simple **occlusion** method (from `src/explain.py`): mask each
resume token one at a time and measure how much the model's confidence in
the predicted class drops. Tokens with the biggest drop are the ones the
model is relying on most.


In [ ]:
explain_result = explain_module.occlusion_importance(
    model, tokenizer, MAX_LEN, sample_resume, sample_jd, top_k=10
)

print(f"Predicted class: {explain_result['predicted_class']}")
print(f"Base confidence: {explain_result['base_confidence'] * 100:.1f}%\n")
print("Most influential resume tokens:")
for token, drop in explain_result["top_influential_tokens"]:
    print(f"  {token:<20s} confidence drop: {drop * 100:+.2f}%")


## 10. Next steps

* Swap `data/raw/train_sample.jsonl` for the **full** `train.jsonl` and
  re-run preprocessing for a larger, higher-fidelity training set.
* Try `Contrastive` or `Triplet` loss (section 10.3 of the design doc) to
  train embeddings directly instead of classification.
* Add attention pooling instead of `GlobalMaxPooling1D` for a learned,
  visualizable attention map.
* Wrap `src/predict.py` in a small Flask/FastAPI service for real-time
  ATS-style scoring.
